# Phase 1 — Data Exploration & Quality Validation

**Project:** Fraud AI Investigator — MENA Fintech Portfolio  
**Notebook:** `notebooks/phase1_data_exploration.ipynb`  
**Author:** Ahmed Raza  
**Last updated:** 2026-05

---

## Objective

Before building any detection logic, we must understand the data we are working with.
This notebook answers three questions:

1. **Is the synthetic dataset structurally correct?** — schema validation, null checks, type checks
2. **Does it reflect realistic UAE/MENA fraud patterns?** — amount distributions, country corridors, time-of-day clustering
3. **What signal do the raw features carry?** — preview which features will drive alert rules in Phase 2

## Outcome

By the end of this notebook you will have:
- Validated data quality across all three datasets (transactions, KYC, sanctions)
- Identified the key statistical separators between normal and suspicious transactions
- Quantified a baseline rule-engine precision/recall to compare against in later weeks
- Saved 3 charts to `doc/Screenshots/` for the project README

## Limitations

- Dataset is **synthetic** (generated by `scripts/generate_data.py`) — distributions are designed, not observed
- Sample size is small (50 transactions, 10 KYC profiles) — statistical conclusions are illustrative only
- Ground-truth `is_flagged` labels are available here for EDA but are **never available in production** — the system must infer fraud from signals alone
- `datetime.utcnow()` deprecation warnings from Python 3.12 are expected and do not affect results

## Prerequisites

```bash
# 1. Generate the datasets (run from project root)
uv run python scripts/generate_data.py

# 2. Install notebook dependencies
uv pip install jupyter pandas matplotlib seaborn

# 3. Launch this notebook
uv run jupyter notebook notebooks/phase1_data_exploration.ipynb
```

---
## Section 0 — Environment setup

**Purpose:** Import all libraries, configure display settings, and add the project root to `sys.path`
so we can import our own modules (e.g. `app.shared.models`) from anywhere in the notebook.

**Why `sys.path.insert`?** Jupyter's working directory is the `notebooks/` folder, not the project root.
Without this, `import app.shared.models` would fail with `ModuleNotFoundError`.
This is the standard pattern for notebooks inside a subdirectory of a Python project.

In [ ]:
# ── Standard library ──────────────────────────────────────────────────────────
import json
import sys
import warnings
from pathlib import Path
from datetime import datetime

# ── Third-party ───────────────────────────────────────────────────────────────
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.patches import Patch
import seaborn as sns

# ── Project root path setup ───────────────────────────────────────────────────
# notebooks/ is one level below the project root, so we go up one directory.
PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

# ── Suppress DeprecationWarnings from datetime.utcnow() in Pydantic/Python 3.12
# These come from our Pydantic models and are non-breaking — safe to suppress here.
warnings.filterwarnings("ignore", category=DeprecationWarning)

# ── Plotting configuration ────────────────────────────────────────────────────
plt.style.use("seaborn-v0_8-whitegrid")
sns.set_palette("husl")
pd.set_option("display.max_columns", 20)   # show all columns in DataFrames
pd.set_option("display.float_format", "{:,.2f}".format)  # comma-formatted floats

# ── Constants ─────────────────────────────────────────────────────────────────
DATA_DIR = PROJECT_ROOT / "app" / "data"
SCREENSHOTS_DIR = PROJECT_ROOT / "doc" / "Screenshots"
SCREENSHOTS_DIR.mkdir(parents=True, exist_ok=True)  # create if missing

# ── Validation: confirm data directory exists before loading ──────────────────
assert DATA_DIR.exists(), (
    f"Data directory not found at {DATA_DIR}. "
    "Run: uv run python scripts/generate_data.py"
)

print(f"Project root  : {PROJECT_ROOT}")
print(f"Data directory: {DATA_DIR}")
print(f"Screenshots   : {SCREENSHOTS_DIR}")
print(f"Environment ready  ✓")
print(f"Run timestamp : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

---
## Section 1 — Data loading & schema validation

**Purpose:** Load all three datasets from JSON and validate that every expected field is
present, correctly typed, and within acceptable ranges.

**Why validate before analysing?** In production, data pipelines fail silently —
a missing field or wrong type can produce misleading statistics rather than an error.
Explicit assertions here catch data generation bugs immediately.

**Design note:** We load raw JSON first (for flexibility), then construct typed
Pydantic models to validate the same data against the production schema.
If the models accept it, the API will accept it.

In [ ]:
# ── Load raw JSON files ───────────────────────────────────────────────────────
required_files = ["transactions.json", "kyc_profiles.json", "sanctions_watchlist.json"]

for fname in required_files:
    path = DATA_DIR / fname
    assert path.exists(), f"Missing: {path} — run scripts/generate_data.py"

with open(DATA_DIR / "transactions.json") as f:
    raw_txs = json.load(f)

with open(DATA_DIR / "kyc_profiles.json") as f:
    raw_kyc = json.load(f)

with open(DATA_DIR / "sanctions_watchlist.json") as f:
    raw_sanctions = json.load(f)

# ── Convert to DataFrames ─────────────────────────────────────────────────────
txs = pd.DataFrame(raw_txs)
kyc = pd.DataFrame(raw_kyc)

# Parse timestamp string → proper datetime column for time-based analysis
txs["timestamp"] = pd.to_datetime(txs["timestamp"], utc=True)
txs["hour"]      = txs["timestamp"].dt.hour   # extract hour for time-of-day analysis
txs["amount_aed"] = txs["amount_aed"].astype(float)  # ensure numeric for plotting

# ── Schema assertions ─────────────────────────────────────────────────────────
# These will raise AssertionError immediately if the data doesn't meet expectations,
# rather than silently producing wrong analysis.

REQUIRED_TX_COLS  = {"tx_id", "customer_id", "amount_aed", "currency", "merchant", "country", "timestamp", "is_flagged"}
REQUIRED_KYC_COLS = {"customer_id", "name", "nationality", "account_age_days", "device_id", "last_known_device", "risk_tier"}

assert REQUIRED_TX_COLS.issubset(txs.columns),  f"Missing transaction columns: {REQUIRED_TX_COLS - set(txs.columns)}"
assert REQUIRED_KYC_COLS.issubset(kyc.columns), f"Missing KYC columns: {REQUIRED_KYC_COLS - set(kyc.columns)}"
assert txs["tx_id"].is_unique,      "FAIL: duplicate transaction IDs detected"
assert txs["amount_aed"].gt(0).all(), "FAIL: non-positive amounts found"
assert txs["currency"].eq("AED").all(), "FAIL: non-AED currency found"
assert txs.isnull().sum().sum() == 0, "FAIL: null values found in transactions"

print("Schema validation passed  ✓")
print(f"Transactions : {len(txs):>4} rows  |  {len(txs.columns)} columns")
print(f"KYC profiles : {len(kyc):>4} rows  |  {len(kyc.columns)} columns")
print(f"Sanctions    : {len(raw_sanctions):>4} entries")

In [ ]:
# ── Validate against production Pydantic models ───────────────────────────────
# This is the key production-readiness check: if our data passes Pydantic validation,
# it will pass the same validation when received by the FastAPI endpoints.

from decimal import Decimal
from app.shared.models import Transaction, KYCProfile, SanctionsEntry

validation_errors = []

for i, row in enumerate(raw_txs):
    try:
        # Pydantic requires Decimal for amount_aed — convert from float
        row_copy = {**row, "amount_aed": Decimal(str(row["amount_aed"]))}
        Transaction(**row_copy)
    except Exception as e:
        validation_errors.append(f"Transaction row {i}: {e}")

for i, row in enumerate(raw_kyc):
    try:
        KYCProfile(**row)
    except Exception as e:
        validation_errors.append(f"KYC row {i}: {e}")

for i, row in enumerate(raw_sanctions):
    try:
        SanctionsEntry(**row)
    except Exception as e:
        validation_errors.append(f"Sanctions row {i}: {e}")

if validation_errors:
    print(f"PYDANTIC VALIDATION ERRORS ({len(validation_errors)}):")
    for err in validation_errors:
        print(f"  {err}")
else:
    print(f"Pydantic model validation passed  ✓")
    print(f"All {len(raw_txs)} transactions, {len(raw_kyc)} KYC profiles, "
          f"and {len(raw_sanctions)} sanctions entries are production-compatible")

---
## Section 2 — Dataset overview

**Purpose:** Understand the basic shape of each dataset before deeper analysis.

**Key question:** Is the fraud rate approximately 20%? This is important because
a rule engine calibrated for 20% fraud behaves very differently at 5% or 50%.
In real UAE AML operations, fraud rates in screened populations are typically 1–5%,
but our synthetic 20% gives richer signal for development purposes.

In [ ]:
# ── Transaction summary statistics ────────────────────────────────────────────
normal     = txs[txs["is_flagged"] == False]
suspicious = txs[txs["is_flagged"] == True]

print("=" * 55)
print("TRANSACTION DATASET SUMMARY")
print("=" * 55)
print(f"Total transactions  : {len(txs)}")
print(f"Normal (not flagged): {len(normal)}  ({len(normal)/len(txs):.0%})")
print(f"Suspicious (flagged): {len(suspicious)}  ({len(suspicious)/len(txs):.0%})")
print()
print("Amount statistics (AED):")
print(f"  Overall  — mean: AED {txs['amount_aed'].mean():>10,.2f}  |  median: AED {txs['amount_aed'].median():>10,.2f}")
print(f"  Normal   — mean: AED {normal['amount_aed'].mean():>10,.2f}  |  max: AED {normal['amount_aed'].max():>10,.2f}")
print(f"  Suspic.  — mean: AED {suspicious['amount_aed'].mean():>10,.2f}  |  min: AED {suspicious['amount_aed'].min():>10,.2f}")
print()
print(f"Date range: {txs['timestamp'].min().date()} → {txs['timestamp'].max().date()}")
print(f"Unique customers: {txs['customer_id'].nunique()}")
print(f"Unique countries: {txs['country'].nunique()} — {sorted(txs['country'].unique())}")

In [ ]:
# ── Full transaction schema with data types ───────────────────────────────────
# Production note: Always inspect dtypes after loading — JSON loads numbers as float64
# by default, which can cause precision issues with financial amounts.
# For production, use Decimal for monetary values (handled in Pydantic models).

print("Column dtypes:")
print(txs.dtypes)
print()
print("First 3 transactions:")
txs[["tx_id", "customer_id", "amount_aed", "currency", "country", "is_flagged"]].head(3)

---
## Section 3 — Amount distribution analysis

**Purpose:** Verify that the synthetic data correctly separates normal vs suspicious transactions
by amount, centred around the UAE Central Bank reporting threshold of AED 40,000.

**UAE regulatory context:** Under CBUAE Anti-Money Laundering guidelines, cash transactions
above AED 40,000 must be reported. This threshold is our primary rule trigger.
A well-designed dataset should show:
- Normal transactions: almost entirely below AED 40,000
- Suspicious transactions: almost entirely above AED 40,000

**Limitation:** The clean separation here is by design. Real-world data has far more
overlap — many legitimate large transactions exist (e.g. property deposits, car purchases),
which is why the LLM triage layer in Phase 3 is needed to reduce false positives.

In [ ]:
# ── Constants referenced in the regulatory context above ─────────────────────
HIGH_VALUE_THRESHOLD_AED = 40_000   # UAE Central Bank cash reporting threshold
HIGH_RISK_COUNTRIES = {"IR", "KP", "SY", "MM", "YE", "SD"}  # FATF high-risk jurisdictions

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, label, df, color in [
    (axes[0], "Normal",     normal,     "#4CAF50"),
    (axes[1], "Suspicious", suspicious, "#F44336"),
]:
    # Plot histogram of transaction amounts
    ax.hist(
        df["amount_aed"],
        bins=20,
        color=color,
        alpha=0.80,
        edgecolor="white",
        linewidth=0.5,
    )

    # Mark the regulatory reporting threshold as a vertical reference line
    ax.axvline(
        x=HIGH_VALUE_THRESHOLD_AED,
        color="crimson",
        linestyle="--",
        linewidth=1.5,
        alpha=0.8,
        label=f"AED {HIGH_VALUE_THRESHOLD_AED:,} CBUAE threshold",
    )

    ax.set_title(f"{label} transactions — AED amount distribution", fontsize=12)
    ax.set_xlabel("Transaction amount (AED)")
    ax.set_ylabel("Number of transactions")
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x/1000:.0f}k"))
    ax.legend(fontsize=9)

    # Annotate with mean value
    mean_val = df["amount_aed"].mean()
    ax.axvline(x=mean_val, color="navy", linestyle=":", linewidth=1.2, alpha=0.6)
    ax.text(mean_val * 1.02, ax.get_ylim()[1] * 0.9, f"mean\n{mean_val/1000:.0f}k",
            color="navy", fontsize=9)

plt.suptitle(
    "Transaction Amount Distribution: Normal vs Suspicious",
    y=1.02, fontsize=13, fontweight="bold"
)
plt.tight_layout()

# Save for README / portfolio documentation
save_path = SCREENSHOTS_DIR / "01_amount_distribution.png"
plt.savefig(save_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Chart saved: {save_path}")

# ── Quantitative summary ──────────────────────────────────────────────────────
pct_normal_below   = (normal["amount_aed"] < HIGH_VALUE_THRESHOLD_AED).mean()
pct_suspic_above   = (suspicious["amount_aed"] > HIGH_VALUE_THRESHOLD_AED).mean()
print(f"\nNormal transactions below AED 40k threshold: {pct_normal_below:.0%}")
print(f"Suspicious transactions above AED 40k threshold: {pct_suspic_above:.0%}")

---
## Section 4 — Country corridor risk analysis

**Purpose:** Verify that suspicious transactions route through FATF-listed high-risk
jurisdictions, and normal transactions stay in standard corridors.

**FATF context:** The Financial Action Task Force (FATF) maintains a grey list and black list
of jurisdictions with inadequate AML/CFT controls. Any transaction with a counterparty
in these countries is automatically elevated in risk. The UAE itself is very focused on
FATF compliance following its own grey-listing in 2022 and subsequent removal in 2024.

**What we expect:** Suspicious transactions cluster in IR (Iran), KP (North Korea),
SY (Syria), MM (Myanmar), YE (Yemen), SD (Sudan) — all under active UN/OFAC sanctions.
Normal transactions cluster in AE (UAE), SA (Saudi Arabia), GB (UK), US (USA).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, label, df in [
    (axes[0], "Normal",     normal),
    (axes[1], "Suspicious", suspicious),
]:
    counts = df["country"].value_counts()

    # Color bars by risk level: red = FATF high-risk, blue = standard
    bar_colors = [
        "#D32F2F" if country in HIGH_RISK_COUNTRIES else "#1565C0"
        for country in counts.index
    ]

    ax.bar(counts.index, counts.values, color=bar_colors, alpha=0.85, edgecolor="white")
    ax.set_title(f"{label} transactions — country of origin", fontsize=12)
    ax.set_xlabel("Country (ISO 3166-1 alpha-2)")
    ax.set_ylabel("Transaction count")

    # Add value labels on top of each bar
    for i, (country, val) in enumerate(counts.items()):
        ax.text(i, val + 0.1, str(val), ha="center", fontsize=10)

    # Legend explaining the colour encoding
    legend_patches = [
        Patch(facecolor="#D32F2F", label="FATF/UN high-risk jurisdiction"),
        Patch(facecolor="#1565C0", label="Standard jurisdiction"),
    ]
    ax.legend(handles=legend_patches, fontsize=9)

plt.suptitle(
    "Transaction Country Profiles: Normal vs Suspicious",
    y=1.02, fontsize=13, fontweight="bold"
)
plt.tight_layout()

save_path = SCREENSHOTS_DIR / "02_country_distribution.png"
plt.savefig(save_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Chart saved: {save_path}")

# ── Sanity check: all suspicious txs should route through high-risk countries ─
suspic_high_risk_pct = suspicious["country"].isin(HIGH_RISK_COUNTRIES).mean()
print(f"\nSuspicious transactions in high-risk corridors: {suspic_high_risk_pct:.0%} (expected ~100%)")
normal_high_risk_pct = normal["country"].isin(HIGH_RISK_COUNTRIES).mean()
print(f"Normal transactions in high-risk corridors   : {normal_high_risk_pct:.0%} (expected ~0%)")

---
## Section 5 — Time-of-day pattern analysis

**Purpose:** Identify whether suspicious transactions cluster at unusual hours,
which is a known fraud signal in AML detection systems.

**Why this matters in production:** Fraudulent actors often initiate transactions
during overnight hours (2–5am local time) when human oversight is minimal and
automated alerts may go unreviewed. UAE financial institutions monitor overnight
transaction spikes as a key AML indicator.

**Limitation:** Our timestamps are UTC and the synthetic generator deliberately
clusters suspicious transactions at hours 2–5 UTC. In production, you would
convert to the customer's local timezone before this analysis.

In [ ]:
fig, ax = plt.subplots(figsize=(13, 4))

all_hours = list(range(24))

# Count transactions per hour for each category, fill 0 for missing hours
normal_by_hour     = normal.groupby("hour").size().reindex(all_hours, fill_value=0)
suspicious_by_hour = suspicious.groupby("hour").size().reindex(all_hours, fill_value=0)

# Stacked bar: normal at bottom, suspicious stacked on top
ax.bar(all_hours, normal_by_hour,     label="Normal",     color="#4CAF50", alpha=0.75)
ax.bar(all_hours, suspicious_by_hour, label="Suspicious", color="#F44336", alpha=0.85,
       bottom=normal_by_hour)

# Highlight the fraud cluster window identified in our data generation
ax.axvspan(2 - 0.5, 5 + 0.5, alpha=0.08, color="crimson",
           label="Synthetic fraud cluster window (2–5am UTC)")

ax.set_title("Transaction volume by hour of day (UTC timestamps)", fontsize=12)
ax.set_xlabel("Hour (UTC, 0 = midnight)")
ax.set_ylabel("Transaction count")
ax.set_xticks(all_hours)
ax.set_xticklabels([f"{h:02d}:00" for h in all_hours], rotation=45, ha="right", fontsize=8)
ax.legend(fontsize=9)

plt.tight_layout()
save_path = SCREENSHOTS_DIR / "03_time_of_day.png"
plt.savefig(save_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Chart saved: {save_path}")

# ── Quantify the concentration ────────────────────────────────────────────────
fraud_in_window = suspicious[suspicious["hour"].between(2, 5)]
print(f"\nSuspicious txs in 2–5am window: {len(fraud_in_window)} "
      f"of {len(suspicious)} ({len(fraud_in_window)/len(suspicious):.0%})")

---
## Section 6 — KYC profile analysis

**Purpose:** Inspect the Know Your Customer (KYC) profiles for risk signals
that the alert engine will use in Phase 2.

**Key signals in KYC data:**
- `has_device_mismatch`: customer's current device ≠ device registered during KYC — strong account takeover signal
- `account_age_days < 30`: new accounts making large transactions — common money mule pattern
- `risk_tier = HIGH`: manually elevated during onboarding due to PEP status, adverse media, or geography

**Production note:** In a real system, KYC data comes from a separate identity
verification service (e.g. Jumio, Onfido) and is joined to transaction data at query time.
We simulate this by storing KYC as a separate JSON file and loading it in the alert engine.

In [ ]:
# ── KYC summary table ─────────────────────────────────────────────────────────
display_cols = ["customer_id", "nationality", "account_age_days", "has_device_mismatch", "risk_tier"]

print("KYC Profile Risk Summary")
print("=" * 55)
print(kyc[display_cols].to_string(index=False))
print()

# ── Aggregate risk signal counts ──────────────────────────────────────────────
n_device_mismatch = kyc["has_device_mismatch"].sum()
n_new_accounts    = (kyc["account_age_days"] < 30).sum()
n_high_risk_tier  = (kyc["risk_tier"] == "HIGH").sum()

print("Risk signal breakdown:")
print(f"  Device mismatches       : {n_device_mismatch} of {len(kyc)} customers ({n_device_mismatch/len(kyc):.0%})")
print(f"  New accounts (<30 days) : {n_new_accounts} of {len(kyc)} customers ({n_new_accounts/len(kyc):.0%})")
print(f"  HIGH risk tier          : {n_high_risk_tier} of {len(kyc)} customers ({n_high_risk_tier/len(kyc):.0%})")

# ── Cross-reference: which transactions belong to KYC-flagged customers? ──────
mismatch_customers = kyc[kyc["has_device_mismatch"]]["customer_id"].tolist()
txs_from_mismatch  = txs[txs["customer_id"].isin(mismatch_customers)]
print(f"\nTransactions from device-mismatch customers: {len(txs_from_mismatch)} "
      f"({len(txs_from_mismatch)/len(txs):.0%} of total)")

---
## Section 7 — Sanctions watchlist inspection

**Purpose:** Review the sanctions watchlist structure and assess its coverage
for the sanctions screening rule in Phase 2.

**Key challenge — Arabic name transliteration:** Arabic names have multiple valid
romanisation forms (Mohamed / Mohammed / Muhammad / Mohamad). Any sanctions
screening system that only matches exact strings will miss obvious variants.
Our watchlist includes aliases for this reason, and Phase 2's sanctions agent
uses fuzzy matching to catch transliteration variants.

**Production note:** Real sanctions lists (OFAC SDN, EU Consolidated List, UN Consolidated)
contain hundreds of thousands of entries and are updated daily. Commercial screening
vendors (Refinitiv, LexisNexis, ComplyAdvantage) are used in production. Our 5-entry
synthetic list demonstrates the mechanics only.

In [ ]:
# ── Inspect each watchlist entry ──────────────────────────────────────────────
print("SANCTIONS WATCHLIST — SYNTHETIC DEMO ENTRIES")
print("=" * 65)
print(f"{'Note: All entities are fictional — for demonstration only':^65}")
print("=" * 65)

for i, entry in enumerate(raw_sanctions, 1):
    print(f"\nEntry {i}: {entry['name']}")
    print(f"  Country : {entry['country']}")
    print(f"  Reason  : {entry['reason']}")
    print(f"  Listed  : {entry.get('date_listed', 'N/A')}")

    # Show alias coverage — critical for Arabic name fuzzy matching
    aliases = entry.get("aliases", [])
    if aliases:
        print(f"  Aliases ({len(aliases)}):")
        for alias in aliases:
            print(f"    • {alias}")
    else:
        print("  Aliases: none — screening relies on exact match only (risky)")

# ── Alias coverage summary ────────────────────────────────────────────────────
print()
total_aliases   = sum(len(e.get("aliases", [])) for e in raw_sanctions)
entries_w_alias = sum(1 for e in raw_sanctions if e.get("aliases"))
print(f"Coverage summary: {entries_w_alias}/{len(raw_sanctions)} entries have aliases "
      f"({total_aliases} aliases total)")
print(f"Screening surface: {len(raw_sanctions) + total_aliases} name variants to match against")

---
## Section 8 — Baseline rule engine performance

**Purpose:** Measure the precision and recall of two simple rules before building
the full alert engine in Phase 2. This creates a quantitative baseline to compare against.

**Metrics explained:**
- **Precision** = of all alerts raised, what fraction were real fraud?  
  High precision = fewer wasted analyst hours on false alarms
- **Recall** = of all real fraud, what fraction did we catch?  
  High recall = fewer fraudulent transactions slipping through undetected
- **F1 Score** = harmonic mean of precision and recall — single number for comparison

**Expected result:** High recall (we catch most fraud), lower precision (some false alarms).
This is the correct trade-off for a first-line rule engine — missing fraud is worse
than over-alerting, because the LLM triage layer in Phase 3 will filter false positives
before they reach a human analyst.

**Limitation:** We evaluate against synthetic `is_flagged` ground truth labels,
which are perfectly correlated with the rules by construction. Real-world evaluation
requires labelled historical data and holdout test sets.

In [ ]:
# ── Rule evaluation ───────────────────────────────────────────────────────────
# Apply the same rules that will run in app/services/alert_engine.py

# Rule 1: Transaction amount above UAE CBUAE reporting threshold
rule_high_value = txs["amount_aed"] > HIGH_VALUE_THRESHOLD_AED

# Rule 2: Transaction routed through FATF high-risk jurisdiction
rule_sanctioned = txs["country"].isin(HIGH_RISK_COUNTRIES)

# Combined alert trigger: either rule fires → alert created
# Note: OR logic here. In Phase 2 each rule creates a separate alert,
# but for evaluation purposes we treat any trigger as a positive prediction.
any_rule_fired = rule_high_value | rule_sanctioned

ground_truth = txs["is_flagged"]

# ── Confusion matrix components ───────────────────────────────────────────────
tp = int((any_rule_fired & ground_truth).sum())   # real fraud, correctly alerted
fp = int((any_rule_fired & ~ground_truth).sum())  # not fraud, incorrectly alerted
fn = int((~any_rule_fired & ground_truth).sum())  # real fraud, missed by rules
tn = int((~any_rule_fired & ~ground_truth).sum()) # not fraud, correctly not alerted

# ── Metric calculation ────────────────────────────────────────────────────────
precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
recall    = tp / (tp + fn) if (tp + fn) > 0 else 0.0
f1        = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
fpr       = fp / (fp + tn) if (fp + tn) > 0 else 0.0  # false positive rate

# ── Results ───────────────────────────────────────────────────────────────────
print("RULE ENGINE BASELINE PERFORMANCE")
print("=" * 45)
print("Rules evaluated:")
print(f"  Rule 1 — amount > AED {HIGH_VALUE_THRESHOLD_AED:,}")
print(f"  Rule 2 — country in FATF high-risk list")
print()
print("Confusion matrix:")
print(f"  True Positives  (TP): {tp:>3}  real fraud correctly alerted")
print(f"  False Positives (FP): {fp:>3}  legitimate tx incorrectly alerted")
print(f"  False Negatives (FN): {fn:>3}  real fraud missed")
print(f"  True Negatives  (TN): {tn:>3}  legitimate tx correctly cleared")
print()
print("Performance metrics:")
print(f"  Precision          : {precision:.1%}  (of alerts raised, % that are real fraud)")
print(f"  Recall             : {recall:.1%}  (of real fraud, % that were caught)")
print(f"  F1 Score           : {f1:.1%}  (harmonic mean of precision and recall)")
print(f"  False Positive Rate: {fpr:.1%}  (of legitimate txs, % incorrectly alerted)")
print()
print("Interpretation:")
if recall >= 0.8:
    print(f"  ✓ High recall ({recall:.0%}) — rules catch most real fraud (desired behaviour)")
else:
    print(f"  ✗ Low recall ({recall:.0%}) — rules are missing real fraud (investigate)")
if precision < 0.7:
    print(f"  ~ Low precision ({precision:.0%}) — expected at rule-engine stage;")
    print(f"    LLM triage (Phase 3) will filter {fp} false positive(s) before analyst review")

# Store baseline for comparison in Phase 2+ notebooks
WEEK1_BASELINE = {"precision": round(precision, 4), "recall": round(recall, 4), "f1": round(f1, 4)}
print(f"\nBaseline stored: {WEEK1_BASELINE}")

---
## Section 9 — Phase 1 summary & sign-off

This section validates all expected outputs before marking the notebook as complete.

In [ ]:
# ── Output validation — confirm all expected artefacts were produced ───────────
expected_charts = [
    SCREENSHOTS_DIR / "01_amount_distribution.png",
    SCREENSHOTS_DIR / "02_country_distribution.png",
    SCREENSHOTS_DIR / "03_time_of_day.png",
]

print("PHASE 1 NOTEBOOK — COMPLETION CHECKLIST")
print("=" * 50)

checks = {
    "Transactions loaded and schema-validated": len(txs) > 0,
    "KYC profiles loaded and schema-validated": len(kyc) > 0,
    "Sanctions watchlist loaded"              : len(raw_sanctions) > 0,
    "Pydantic model validation passed"        : len(validation_errors) == 0,
    "Amount distribution chart saved"        : expected_charts[0].exists(),
    "Country distribution chart saved"       : expected_charts[1].exists(),
    "Time-of-day chart saved"                : expected_charts[2].exists(),
    "Baseline metrics computed"              : bool(WEEK1_BASELINE),
    "High recall achieved (≥80%)": WEEK1_BASELINE["recall"] >= 0.80,
}

all_passed = True
for label, passed in checks.items():
    status = "✓" if passed else "✗"
    print(f"  {status}  {label}")
    if not passed:
        all_passed = False

print()
if all_passed:
    print("All checks passed — Phase 1 notebook complete  ✓")
    print("Ready to build the alert engine in Phase 2.")
else:
    print("Some checks failed — review sections above before proceeding.")

print(f"\nCompleted at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")